In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Encoder(nn.Module):
    def __init__(self, embedding_dim=16):
        super().__init__()

        self.layers = nn.Sequential(
            # [B, 1, 28, 28] -> [B, 32, 14, 14]
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.GELU(),

            # [B, 32, 14, 14] -> [B, embedding_dim, 7, 7]
            nn.Conv2d(32, embedding_dim, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x):
        return self.layers(x)


class Decoder(nn.Module):
    def __init__(self, embedding_dim=16):
        super().__init__()

        self.layers = nn.Sequential(
            # [B, embedding_dim, 7, 7] -> [B, 32, 14, 14]
            nn.ConvTranspose2d(
                embedding_dim, 32, kernel_size=4, stride=2, padding=1
            ),
            nn.GELU(),

            # [B, 32, 14, 14] -> [B, 1, 28, 28]
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.layers(z)


class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=64, embedding_dim=16, beta=0.25):
        super().__init__()

        self.beta = beta
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)

        nn.init.uniform_(
            self.codebook.weight,
            -1 / num_embeddings,
            1 / num_embeddings,
        )

    def forward(self, z):
        # Move channels last: [B, D, H, W] -> [B, H, W, D]
        z = z.permute(0, 2, 3, 1).contiguous()
        flat_z = z.reshape(-1, z.shape[-1])

        # つむぎ：ここの距離の二乗の計算はコツが必要
        distances = (
            flat_z.square().sum(dim=1, keepdim=True)
            + self.codebook.weight.square().sum(dim=1)
            - 2 * flat_z @ self.codebook.weight.T
        )

        # Choose the nearest codebook vector for each position.
        indices = distances.argmin(dim=1)
        quantized = self.codebook(indices).view_as(z)

        # Train the codebook and encourage encoder commitment.
        codebook_loss = F.mse_loss(quantized, z.detach())
        commitment_loss = F.mse_loss(z, quantized.detach())
        vq_loss = codebook_loss + self.beta * commitment_loss

        # Straight　Throughという手法が使われる。
        quantized = z + (quantized - z).detach()

        # Restore shape: [B, H, W, D] -> [B, D, H, W]
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        return quantized, vq_loss


class VQVAE(nn.Module):
    def __init__(self, num_embeddings=64, embedding_dim=16):
        super().__init__()

        self.encoder = Encoder(embedding_dim)
        self.quantizer = VectorQuantizer(num_embeddings, embedding_dim)
        self.decoder = Decoder(embedding_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss = self.quantizer(z)
        reconstructed = self.decoder(quantized)

        return reconstructed, vq_loss

In [ ]:
import torch
from torch import nn

class Encoder(nn.Module):
    def __init__(self, in_channels, num_hiddens, embedding_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, num_hiddens // 2, kernel_size=4, stride=2, padding=1),  # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(num_hiddens // 2, num_hiddens, kernel_size=4, stride=2, padding=1),  # 14x14 -> 7x7
            nn.ReLU(),
            nn.Conv2d(num_hiddens, embedding_dim, kernel_size=3, stride=1, padding=1)     # 7x7 -> 7x7
        )

    def forward(self, x):
        return self.conv(x)


class 簡単VAE(nn.Module):
    def __init__(self, in_channels, latent_space):
        super().__init__()
        
        self.encoder = nn.Linear(in_channels, latent_space)
        self.decoder = nn.Linear(latent_space, in_channels)  # 入力と逆
        
    def forward(self, x):
        latent = self.encoder(x)
        print("-> 潜在空間：", latent.shape)
        y = self.decoder(latent)
        return y
    
model = 簡単VAE(16, 8)
rand_input = torch.rand((16))

print("-> 入力　　：", rand_input.shape)
out = model(rand_input)
print("-> 出力　　：", out.shape)

-> 入力　　： torch.Size([16])
-> 潜在空間： torch.Size([8])
-> 出力　　： torch.Size([16])
